# Amortization `tuplesPerSecond` comparison

Compare the per-run-averaged `tuplesPerSecond` between two amortization result CSVs.

Both files share the same parameter schema. For every parameter combination we average the
metric across its runs (`run_idx`) in each file, then report the **difference of those per-run
averages** (`current - baseline`).

- **baseline** = `results_synopsis_amortization.csv.bak`
- **current**  = `results_synopsis_amortization.csv`


In [1]:
import sys
from pathlib import Path

import pandas as pd


## Configuration
Edit these to point at different files / metric.

In [2]:
# Resolve the directory the CSVs live in. Falls back to the current working dir
# when __file__ is unavailable (i.e. running interactively in Jupyter).
try:
    HERE = Path(__file__).resolve().parent
except NameError:
    HERE = Path.cwd()

CURRENT = HERE / "results_synopsis_amortization.csv"       # newer results
BASELINE = HERE / "results_synopsis_amortization.csv.bak"  # baseline results
METRIC = "tuplesPerSecond"   # partial match ok; resolves to tuplesPerSecond_listener
TOP = 20                      # how many largest-magnitude diffs to display
OUT = None                   # set to a Path to write the full per-group diff table as CSV

# Columns that identify a single measurement run rather than a parameter, plus the
# per-run outputs. Everything else is treated as a grouping parameter.
NON_PARAM_COLS = {"run_idx", "build_duration_s", "issue"}


## Helper functions

In [3]:
def resolve_metric(columns, requested):
    """Return the actual column for `requested`, allowing a partial/prefix match."""
    if requested in columns:
        return requested
    matches = [c for c in columns if requested in c]
    if len(matches) == 1:
        return matches[0]
    if not matches:
        raise ValueError(f"no column matching '{requested}' in {list(columns)}")
    raise ValueError(f"'{requested}' is ambiguous, matches {matches}; set METRIC explicitly")


def per_group_mean(path, metric, group_cols):
    """Average `metric` across runs for each parameter group in one file."""
    df = pd.read_csv(path)
    df[metric] = pd.to_numeric(df[metric], errors="coerce")
    # Drop rows without a valid measurement (e.g. exception/timeout/crashed runs).
    df = df.dropna(subset=[metric])
    grouped = df.groupby(group_cols, dropna=False)[metric]
    return grouped.agg(mean="mean", runs="count").reset_index()


## Load files and resolve the metric / grouping columns

In [4]:
cur_cols = pd.read_csv(CURRENT, nrows=0).columns
bak_cols = pd.read_csv(BASELINE, nrows=0).columns
if list(cur_cols) != list(bak_cols):
    raise ValueError(f"column schemas differ\n  current : {list(cur_cols)}\n  baseline: {list(bak_cols)}")

metric = resolve_metric(cur_cols, METRIC)
group_cols = [c for c in cur_cols if c not in NON_PARAM_COLS and c != metric]
if METRIC != metric:
    print(f"Using metric column '{metric}' for requested '{METRIC}'.")
print(f"Grouping parameters: {group_cols}")


Using metric column 'tuplesPerSecond_listener' for requested 'tuplesPerSecond'.
Grouping parameters: ['dataset', 'statistic_type', 'memory_budget', 'build_window_size_sec', 'executionMode', 'numberOfWorkerThreads', 'buffersInGlobalBufferManager', 'joinStrategy', 'bufferSizeInBytes', 'pageSize', 'enableLatency', 'statisticStoreType', 'num_synopses', 'query_name']


## Compute per-group run averages and merge

In [5]:
cur = per_group_mean(CURRENT, metric, group_cols)
bak = per_group_mean(BASELINE, metric, group_cols)

merged = cur.merge(bak, on=group_cols, how="outer", suffixes=("_current", "_baseline"), indicator=True)

only_current = merged[merged["_merge"] == "left_only"]
only_baseline = merged[merged["_merge"] == "right_only"]
common = merged[merged["_merge"] == "both"].copy()

common["diff"] = common["mean_current"] - common["mean_baseline"]
common["pct_diff"] = common["diff"] / common["mean_baseline"] * 100.0

print(f"Parameter groups: {len(common)} common, "
      f"{len(only_current)} only in current, {len(only_baseline)} only in baseline")


Parameter groups: 44 common, 8 only in current, 550 only in baseline


## Per-group difference of per-run averages (current - baseline)
Sorted by largest magnitude. `runs_*` shows how many runs each average is over.

In [6]:
label_col = "query_name" if "query_name" in group_cols else None
show = common.reindex(common["diff"].abs().sort_values(ascending=False).index)
cols = ([label_col] if label_col else group_cols) + [
    "mean_baseline", "mean_current", "diff", "pct_diff", "runs_baseline", "runs_current"]

with pd.option_context("display.max_rows", None, "display.width", 200,
                       "display.float_format", lambda v: f"{v:,.2f}"):
    display(show.head(TOP)[cols])
if len(show) > TOP:
    print(f"... ({len(show) - TOP} more groups; raise TOP or set OUT to write all)")


,query_name,mean_baseline,mean_current,diff,pct_diff,runs_baseline,runs_current
402,EquiWidthHistogramAmort_Manufacturing_N2_mb102...,"95,223,666.67","78,712,388.89","-16,511,277.78",-17.34,3.00,3.00
401,EquiWidthHistogramAmort_Manufacturing_N1_mb102...,"95,637,166.67","105,147,833.33","9,510,666.67",9.94,3.00,3.00
175,EquiWidthHistogramAmort_ClusterMonitoring_N1_m...,"58,434,222.22","61,952,666.67","3,518,444.44",6.02,3.00,3.00
403,EquiWidthHistogramAmort_Manufacturing_N4_mb102...,"63,701,333.33","60,930,666.67","-2,770,666.67",-4.35,3.00,3.00
85,CountMinAmort_ClusterMonitoring_N1_mb10240_10sec,"56,236,333.33","53,734,583.33","-2,501,750.00",-4.45,3.00,3.00
211,ReservoirAmort_ClusterMonitoring_N1_mb10240_10sec,"44,614,583.33","46,384,305.56","1,769,722.22",3.97,3.00,3.00
404,EquiWidthHistogramAmort_Manufacturing_N8_mb102...,"32,563,333.33","34,009,777.78","1,446,444.44",4.44,3.00,3.00
86,CountMinAmort_ClusterMonitoring_N2_mb10240_10sec,"31,430,166.67","32,863,500.00","1,433,333.33",4.56,3.00,3.00
437,ReservoirAmort_Manufacturing_N1_mb10240_10sec,"57,118,555.56","58,518,666.67","1,400,111.11",2.45,3.00,3.00
397,EquiWidthHistogramAmort_Manufacturing_N2_mb102...,"16,001,976.98","14,983,425.75","-1,018,551.24",-6.37,3.00,3.00


... (24 more groups; raise TOP or set OUT to write all)


## Summary across common parameter groups

In [7]:
print(f"  groups compared          : {len(common)}")
print(f"  mean diff (current-base) : {common['diff'].mean():,.2f} tuples/s")
print(f"  median diff              : {common['diff'].median():,.2f} tuples/s")
print(f"  mean |diff|              : {common['diff'].abs().mean():,.2f} tuples/s")
print(f"  mean pct diff            : {common['pct_diff'].mean():+.2f} %")
print(f"  median pct diff          : {common['pct_diff'].median():+.2f} %")
print(f"  groups current > baseline: {(common['diff'] > 0).sum()}")
print(f"  groups current < baseline: {(common['diff'] < 0).sum()}")


  groups compared          : 44
  mean diff (current-base) : -88,747.72 tuples/s
  median diff              : 5,678.75 tuples/s
  mean |diff|              : 1,114,308.00 tuples/s
  mean pct diff            : +0.15 %
  median pct diff          : +0.26 %
  groups current > baseline: 24
  groups current < baseline: 20


## Parameter groups present in only one file

In [8]:
lbl = "query_name" if "query_name" in group_cols else group_cols[0]
if not only_current.empty:
    names = sorted(only_current[lbl].dropna().unique().tolist())
    print(f"only in current ({len(only_current)}): {names[:10]}{' ...' if len(only_current) > 10 else ''}")
if not only_baseline.empty:
    names = sorted(only_baseline[lbl].dropna().unique().tolist())
    print(f"only in baseline ({len(only_baseline)}): {names[:10]}{' ...' if len(only_baseline) > 10 else ''}")
if only_current.empty and only_baseline.empty:
    print("All parameter groups are present in both files.")


only in current (8): ['PassthroughAmort_ClusterMonitoring_N1_mb10240_10sec', 'PassthroughAmort_Manufacturing_N1_mb10240_10sec', 'SumAmort_ClusterMonitoring_N1_mb10240_10sec', 'SumAmort_Manufacturing_N1_mb10240_10sec']
only in baseline (550): ['CountMinAmort_ClusterMonitoring_N10_mb10240_1sec', 'CountMinAmort_ClusterMonitoring_N10_mb10240_5sec', 'CountMinAmort_ClusterMonitoring_N10_mb1024_10sec', 'CountMinAmort_ClusterMonitoring_N10_mb1024_1sec', 'CountMinAmort_ClusterMonitoring_N10_mb1024_5sec', 'CountMinAmort_ClusterMonitoring_N10_mb5120_10sec', 'CountMinAmort_ClusterMonitoring_N10_mb5120_1sec', 'CountMinAmort_ClusterMonitoring_N10_mb5120_5sec', 'CountMinAmort_ClusterMonitoring_N1_mb10240_1sec', 'CountMinAmort_ClusterMonitoring_N1_mb10240_5sec'] ...


## Optional: write the full per-group diff table

In [9]:
if OUT is not None:
    common.drop(columns="_merge").to_csv(OUT, index=False)
    print(f"Wrote per-group diff table to {OUT}")
else:
    print("OUT is None; set it above to export the full table.")


OUT is None; set it above to export the full table.
